# Pipeline v2 Apps Workbench

Notebook orchestrator for the current pure `src.pipeline_v2.apps` modules.

Each app stays JSON/dict-in and JSON/dict-out. This notebook builds app inputs, calls app entrypoints, and writes notebook-owned outputs under `output_v2/`. Manual ChatGPT handoffs and media voiceover runs live in `v2_chatgpt_voiceover_workbench.ipynb`.


In [1]:
from __future__ import annotations

from copy import deepcopy
from datetime import datetime, timedelta, timezone
import json
from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "pipeline_v2" / "apps").exists():
    raise RuntimeError("Run this notebook from the repository root or output_v2 directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

APPS_DIR = PROJECT_ROOT / "src" / "pipeline_v2" / "apps"
NEWS_COLLECTION_DIR = APPS_DIR / "news_collection"
PIPELINE_V2_CONFIG_PATH = PROJECT_ROOT / "src" / "pipeline_v2" / "config" / "workbench.yaml"
OUTPUT_V2 = PROJECT_ROOT / "output_v2"
IST = timezone(timedelta(hours=5, minutes=30))

print(f"Project root: {PROJECT_ROOT}")
print(f"Apps dir:     {APPS_DIR}")
print(f"Output root:  {OUTPUT_V2}")
print(f"Config:       {PIPELINE_V2_CONFIG_PATH}")

Project root: /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation
Apps dir:     /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/src/pipeline_v2/apps
Output root:  /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2
Config:       /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/src/pipeline_v2/config/workbench.yaml


In [2]:
# Change RUN_PROFILE here for a different v2 app run.
def _load_workbench_config(path: Path) -> dict:
    data = yaml.safe_load(path.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise ValueError(f"Expected YAML object at {path}")
    return data


WORKBENCH_CONFIG = _load_workbench_config(PIPELINE_V2_CONFIG_PATH)
RUN_PROFILE = WORKBENCH_CONFIG.get("default_run_profile", "daily_48h_all_sources")
RUN_ID = f"v2_apps_{datetime.now(IST).strftime('%Y_%m_%d_%H%M%S')}_{RUN_PROFILE}"
PROFILE_CONFIG = deepcopy(WORKBENCH_CONFIG.get("profiles", {}).get(RUN_PROFILE, {}))
if not PROFILE_CONFIG:
    raise KeyError(f"Unknown or empty v2 workbench profile: {RUN_PROFILE}")

NEWS_OUTPUT_DIR = OUTPUT_V2 / "news_collection" / RUN_ID
EVIDENCE_OUTPUT_DIR = OUTPUT_V2 / "evidence_prep" / RUN_ID
STORY_OUTPUT_DIR = OUTPUT_V2 / "story_mapping" / RUN_ID
VIDEO_OUTPUT_DIR = OUTPUT_V2 / "video_packaging" / RUN_ID
SCRIPT_OUTPUT_DIR = OUTPUT_V2 / "script_generation" / RUN_ID
METADATA_OUTPUT_DIR = OUTPUT_V2 / "metadata_creator" / RUN_ID

print(f"Run profile: {RUN_PROFILE}")
print(f"Run ID:      {RUN_ID}")
print(
    "LLMs:        "
    f"story_mapping={PROFILE_CONFIG['story_mapping']['story_label']['provider']}/{PROFILE_CONFIG['story_mapping']['story_label']['model']}, "
    f"script_generation={PROFILE_CONFIG['script_generation']['provider']}/{PROFILE_CONFIG['script_generation']['model']}, "
    f"metadata_creator={PROFILE_CONFIG['metadata_creator']['provider']}/{PROFILE_CONFIG['metadata_creator']['model']}"
)


Run profile: daily_48h_all_sources
Run ID:      v2_apps_2026_06_26_231604_daily_48h_all_sources
LLMs:        story_mapping=openai/gpt-4o-mini, script_generation=openai/gpt-4o, metadata_creator=openai/gpt-4o-mini


In [3]:
def load_yaml(path: Path) -> dict:
    data = yaml.safe_load(path.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise ValueError(f"Expected YAML object at {path}")
    return data


def workbench_module_config(module_name: str) -> dict:
    config = deepcopy(PROFILE_CONFIG.get(module_name, {}))
    if not config:
        raise KeyError(f"Profile {RUN_PROFILE!r} is missing config for {module_name!r}")
    return config


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    text = "\n".join(json.dumps(row, ensure_ascii=False) for row in rows)
    if text:
        text += "\n"
    path.write_text(text, encoding="utf-8")


def require_app_ready(result: dict, app_name: str, *, allowed_statuses=("ok",), handoff_key=None) -> None:
    status = result.get("status")
    if status not in allowed_statuses:
        summary = result.get("summary", {})
        failure_reason = summary.get("failure_reason") or summary.get("error") or result.get("reason") or "No failure reason provided."
        raise RuntimeError(f"{app_name} is not ready. status={status!r}. reason={failure_reason}")
    if handoff_key and not bool(result.get("handoff", {}).get(handoff_key, False)):
        summary = result.get("summary", {})
        failure_reason = summary.get("failure_reason") or summary.get("error") or f"handoff.{handoff_key} is false"
        raise RuntimeError(f"{app_name} is not ready. status={status!r}. reason={failure_reason}")


def require_non_empty_script(script_generation_result: dict) -> str:
    require_app_ready(script_generation_result, "script_generation", handoff_key="ready_for_verification")
    script_package = script_generation_result.get("data", {}).get("script_package", {})
    full_script = str(script_package.get("full_script", "")).strip()
    if not full_script:
        checks = script_package.get("checks", {})
        errors = checks.get("errors") or []
        reason = "; ".join(str(error) for error in errors) or "script_package.full_script is empty"
        raise RuntimeError(f"script_generation did not produce a non-empty full_script. reason={reason}")
    return full_script


def collection_window_for_profile(profile: dict) -> dict:
    end = datetime.now(IST)
    lookback_hours = int(profile["lookback_hours"])
    start = end - timedelta(hours=lookback_hours)
    return {
        "start": start.isoformat(),
        "end": end.isoformat(),
        "lookback_hours": lookback_hours,
    }


def build_news_collection_input(run_id: str, run_profile: str) -> dict:
    profile_config = load_yaml(NEWS_COLLECTION_DIR / "profiles.yaml")
    source_config = load_yaml(NEWS_COLLECTION_DIR / "source_config.yaml")
    profiles = profile_config["profiles"]
    if run_profile not in profiles:
        raise KeyError(f"Unknown run profile: {run_profile}")

    profile = profiles[run_profile]
    source_groups = source_config["layer_1"]["source_groups"]
    window = collection_window_for_profile(profile)
    collectors = {}
    for group_name in profile["enabled_source_groups"]:
        collector_config = deepcopy(source_groups[group_name])
        collector_config["source_group"] = group_name
        collector_config["collection_window"] = window
        collector_config["query_themes"] = list(profile["query_themes"])
        collector_config["runtime"] = deepcopy(profile.get("collector_caps", {}).get(group_name, {}))
        collectors[group_name] = collector_config

    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": {
            "enabled_source_groups": list(profile["enabled_source_groups"]),
            "query_themes": list(profile["query_themes"]),
            "collectors": collectors,
        },
        "input": {
            "run_profile": run_profile,
            "channel": deepcopy(profile_config["channel"]),
        },
    }


def build_evidence_prep_input(run_id: str, news_input: dict, news_result: dict) -> dict:
    first_collector = next(iter(news_input["config"]["collectors"].values()))
    config = workbench_module_config("evidence_prep")
    config["collection_window"] = deepcopy(first_collector["collection_window"])
    config["query_themes"] = list(news_input["config"]["query_themes"])
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": config,
        "input": {
            "news_collection": news_result,
        },
    }


def build_story_mapping_input(run_id: str, evidence_input: dict, evidence_result: dict) -> dict:
    config = workbench_module_config("story_mapping")
    config["collection_window"] = deepcopy(evidence_input["config"]["collection_window"])
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": config,
        "input": {
            "evidence_prep": evidence_result,
        },
    }


def build_video_packaging_input(run_id: str, story_mapping_result: dict) -> dict:
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": workbench_module_config("video_packaging"),
        "input": {
            "story_mapping": story_mapping_result,
        },
    }


def build_script_generation_input(run_id: str, video_packaging_result: dict) -> dict:
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": workbench_module_config("script_generation"),
        "input": {
            "video_packaging": video_packaging_result,
        },
    }


def build_metadata_creator_input(run_id: str, script_generation_result: dict, video_packaging_result: dict) -> dict:
    profile_config = load_yaml(NEWS_COLLECTION_DIR / "profiles.yaml")
    channel = deepcopy(profile_config.get("channel", {}))
    config = workbench_module_config("metadata_creator")
    config["channel"] = {
        "channel_name": channel.get("channel_name", "ManiVerse - AI"),
        "content_promise": channel.get("content_promise", ""),
    }
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": config,
        "input": {
            "script_generation": script_generation_result,
            "video_packaging": video_packaging_result,
        },
    }


## App 1: News Collection

In [4]:
from src.pipeline_v2.apps.news_collection import run as run_news_collection

news_input = build_news_collection_input(RUN_ID, RUN_PROFILE)
news_result = run_news_collection(news_input)
require_app_ready(news_result, "news_collection", allowed_statuses=("ok", "partial"))

write_json(NEWS_OUTPUT_DIR / "news_collection_result.json", news_result)
write_json(NEWS_OUTPUT_DIR / "raw_records.json", news_result["data"])
write_json(NEWS_OUTPUT_DIR / "summary.json", news_result["summary"])

print(f"News collection status: {news_result['status']}")
print(f"Raw records:            {news_result['data']['record_count']}")
print(f"Wrote:                  {NEWS_OUTPUT_DIR}")

News collection status: partial
Raw records:            142
Wrote:                  /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/news_collection/v2_apps_2026_06_26_231604_daily_48h_all_sources


In [5]:
news_result["summary"]

{'run_id': 'v2_apps_2026_06_26_231604_daily_48h_all_sources',
 'generated_at': '2026-06-26T23:20:25.429130+05:30',
 'status': 'partial',
 'enabled_source_groups': ['ai_company_blogs',
  'tech_news_apis',
  'research_papers',
  'youtube_podcast_transcripts',
  'community_signals'],
 'enabled_source_group_count': 5,
 'successful_source_groups': ['ai_company_blogs',
  'tech_news_apis',
  'research_papers',
  'youtube_podcast_transcripts',
  'community_signals'],
 'successful_source_group_count': 5,
 'total_records': 142,
 'elapsed_seconds': 260.13,
 'groups': {'ai_company_blogs': {'collector': 'company_blogs',
   'status': 'partial',
   'fetched_count': 77,
   'collected_count': 67,
   'filtered_out_count': 10,
   'sources': {'openai_news': {'status': 'ok',
     'fetched_count': 4,
     'collected_count': 4,
     'filtered_out_count': 0,
     'notes': ['Content integrity title_mismatch: Previewing GPT-5.6 Sol: a next-generation model (https://openai.com/index/previewing-gpt-5-6-sol)']},
 

## App 2: Evidence Prep

In [6]:
from src.pipeline_v2.apps.evidence_prep import run as run_evidence_prep

evidence_input = build_evidence_prep_input(RUN_ID, news_input, news_result)
evidence_result = run_evidence_prep(evidence_input)
require_app_ready(evidence_result, "evidence_prep")

write_json(EVIDENCE_OUTPUT_DIR / "evidence_prep_result.json", evidence_result)
write_json(EVIDENCE_OUTPUT_DIR / "prepared_records.json", evidence_result["data"])
write_json(EVIDENCE_OUTPUT_DIR / "summary.json", evidence_result["summary"])

print(f"Evidence prep status: {evidence_result['status']}")
print(f"Prepared records:     {evidence_result['data']['record_count']}")
print(f"Wrote:                {EVIDENCE_OUTPUT_DIR}")

Evidence prep status: ok
Prepared records:     54
Wrote:                /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/evidence_prep/v2_apps_2026_06_26_231604_daily_48h_all_sources


In [7]:
evidence_result["summary"]

{'run_id': 'v2_apps_2026_06_26_231604_daily_48h_all_sources',
 'generated_at': '2026-06-26T23:20:25.641536+05:30',
 'status': 'ok',
 'input_record_count': 142,
 'prepared_record_count': 54,
 'excluded_count': 88,
 'duplicate_count': 0,
 'content_status_counts': {'full': 54, 'partial': 0, 'title_only': 0},
 'exclude_reason_counts': {'official_content_title_mismatch': 1,
  'outside_manifest_window': 81,
  'official_content_too_short': 1,
  'off_topic_crypto': 4,
  'promotional_or_deal': 1},
 'source_groups': {'ai_company_blogs': {'input': 67,
   'prepared': 12,
   'duplicates': 0,
   'excluded': 55},
  'tech_news_apis': {'input': 56,
   'prepared': 24,
   'duplicates': 0,
   'excluded': 32},
  'research_papers': {'input': 10,
   'prepared': 10,
   'duplicates': 0,
   'excluded': 0},
  'youtube_podcast_transcripts': {'input': 5,
   'prepared': 5,
   'duplicates': 0,
   'excluded': 0},
  'community_signals': {'input': 4,
   'prepared': 3,
   'duplicates': 0,
   'excluded': 1}},
 'excluded_

## App 3: Story Mapping

In [8]:
# Optional story-mapping resume helper:
# Keep this commented for normal full-notebook runs.
#
# import json
# RESUME_RUN_ID = "v2_apps_YYYY_MM_DD_HHMMSS_daily_48h_all_sources"
# RUN_ID = RESUME_RUN_ID
# with open(PROJECT_ROOT / "output_v2" / "evidence_prep" / RUN_ID / "evidence_prep_result.json", encoding="utf-8") as f:
#     evidence_result = json.load(f)


In [9]:
from src.pipeline_v2.apps.story_mapping import run as run_story_mapping

story_mapping_input = build_story_mapping_input(RUN_ID, evidence_input, evidence_result)
story_mapping_result = run_story_mapping(story_mapping_input)
require_app_ready(story_mapping_result, "story_mapping", handoff_key="ready_for_fact_packet")

write_json(STORY_OUTPUT_DIR / "story_mapping_result.json", story_mapping_result)
write_json(STORY_OUTPUT_DIR / "stories.json", story_mapping_result["data"])
write_json(STORY_OUTPUT_DIR / "summary.json", story_mapping_result["summary"])
write_jsonl(STORY_OUTPUT_DIR / "llm_calls.jsonl", story_mapping_result["llm_calls"])

print(f"Story mapping status: {story_mapping_result['status']}")
print(f"Stories:              {story_mapping_result['data']['story_count']}")
print(f"Wrote:                {STORY_OUTPUT_DIR}")

/Users/manikantamb/Library/Python/3.14/lib/python/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/manikantamb/Library/Python/3.14/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7654.11it/s]


Story mapping status: ok
Stories:              54
Wrote:                /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/story_mapping/v2_apps_2026_06_26_231604_daily_48h_all_sources


In [10]:
story_mapping_result["summary"]

{'run_id': 'v2_apps_2026_06_26_231604_daily_48h_all_sources',
 'generated_at': '2026-06-26T23:20:55.428199+05:30',
 'status': 'ok',
 'input_record_count': 54,
 'story_count': 54,
 'single_record_story_count': 54,
 'multi_record_story_count': 0,
 'semantic_merge_count': 0,
 'dropped_singleton_count': 0,
 'effective_max_pair_hours': 48.0,
 'priority_bucket_counts': {'medium': 20, 'low': 34},
 'source_group_mix': {'ai_company_blogs': 12,
  'tech_news_apis': 24,
  'research_papers': 10,
  'community_signals': 3,
  'youtube_podcast_transcripts': 5},
 'llm_runtime': {'use_llm_story_labels': True,
  'story_label_provider': 'openai',
  'story_label_model': 'gpt-4o-mini',
  'story_label_prompt_version': 'v1',
  'story_label_worker_count': 10,
  'llm_story_label_count': 54,
  'parallelized_steps': ['story_label'],
  'serial_steps': []},
 'top_story_quality_flags': {'low_editorial_value_count': 0,
  'tutorial_count': 0,
  'weak_content_count': 0,
  'event_update_count': 0},
 'top_stories': [{'ran

## App 4: Video Packaging

In [11]:
# Optional video-packaging resume helper:
# Keep this commented for normal full-notebook runs.
#
# import json
# RESUME_RUN_ID = "v2_apps_YYYY_MM_DD_HHMMSS_daily_48h_all_sources"
# RUN_ID = RESUME_RUN_ID
# with open(PROJECT_ROOT / "output_v2" / "story_mapping" / RUN_ID / "story_mapping_result.json", encoding="utf-8") as f:
#     story_mapping_result = json.load(f)


In [12]:
from src.pipeline_v2.apps.video_packaging import run as run_video_packaging

video_packaging_input = build_video_packaging_input(RUN_ID, story_mapping_result)
video_packaging_result = run_video_packaging(video_packaging_input)
require_app_ready(video_packaging_result, "video_packaging", handoff_key="ready_for_script_generation")

write_json(VIDEO_OUTPUT_DIR / "video_packaging_result.json", video_packaging_result)
write_json(VIDEO_OUTPUT_DIR / "video_packet.json", video_packaging_result["data"]["video_packet"])
write_json(VIDEO_OUTPUT_DIR / "summary.json", video_packaging_result["summary"])

print(f"Video packaging status: {video_packaging_result['status']}")
print(f"Selected format:        {video_packaging_result['data']['selected_format']}")
print(f"Selected stories:       {video_packaging_result['summary']['selected_story_count']}")
print(f"Ready for script gen:   {video_packaging_result['handoff']['ready_for_script_generation']}")
print(f"Wrote:                  {VIDEO_OUTPUT_DIR}")

Video packaging status: ok
Selected format:        roundup
Selected stories:       7
Ready for script gen:   True
Wrote:                  /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/video_packaging/v2_apps_2026_06_26_231604_daily_48h_all_sources


In [13]:
video_packaging_result["summary"]

{'input_story_count': 54,
 'candidate_story_count': 20,
 'selected_story_count': 7,
 'rejected_story_count': 47,
 'selection_reason_counts': {'outside_candidate_limit': 34,
  'entity_diversity_cap': 1,
  'after_max_stories': 12},
 'role_counts': {'lead': 1, 'major': 2, 'brief': 4},
 'target_duration_seconds': 480,
 'failure_reason': '',
 'selected_story_ids': ['story_20260625_retrofit_don_t_rebuild_agentic_overlay_ce07e8',
  'story_20260625_patronus_ai_lands_50m_to_build_digital_f0f086',
  'story_20260625_the_white_house_is_asking_openai_to_slow_eb94a0',
  'story_20260626_openai_unveils_gpt_5_6_amid_us_ai_regula_0dfceb',
  'story_20260626_how_cara_pioneers_domain_specific_ai_for_102f98',
  'story_20260626_creating_the_nvidia_nemotron_3_ultra_nvf_cdf42a',
  'story_20260626_it_s_not_about_anthropic_vs_openai_anym_027b18'],
 'rejected_story_ids': {'story_20260626_openai_s_jalape_o_chip_is_big_tech_s_spi_32e17f': ['entity_diversity_cap'],
  'story_20260626_why_everyone_from_openai_to_space

## App 5: Script Generation


In [14]:
# Optional resume helper:
# If you want to start from an existing video_packaging result, set RESUME_RUN_ID
# manually and load it here. Keep this commented for normal full-notebook runs.
#
# import json
# RESUME_RUN_ID = "v2_apps_YYYY_MM_DD_HHMMSS_daily_48h_all_sources"
# RUN_ID = RESUME_RUN_ID
# with open(PROJECT_ROOT / "output_v2" / "video_packaging" / RUN_ID / "video_packaging_result.json", encoding="utf-8") as f:
#     video_packaging_result = json.load(f)


In [15]:
from src.pipeline_v2.apps.script_generation.formats.roundup import run as run_roundup_script_generation

script_generation_input = build_script_generation_input(RUN_ID, video_packaging_result)
script_generation_result = run_roundup_script_generation(script_generation_input)
require_non_empty_script(script_generation_result)
script_package = script_generation_result["data"]["script_package"]

write_json(SCRIPT_OUTPUT_DIR / "script_generation_result.json", script_generation_result)
write_json(SCRIPT_OUTPUT_DIR / "script_package.json", script_package)
write_json(SCRIPT_OUTPUT_DIR / "summary.json", script_generation_result["summary"])
write_jsonl(SCRIPT_OUTPUT_DIR / "llm_calls.jsonl", script_generation_result["llm_calls"])
(SCRIPT_OUTPUT_DIR / "full_script.txt").parent.mkdir(parents=True, exist_ok=True)
(SCRIPT_OUTPUT_DIR / "full_script.txt").write_text(script_package["full_script"], encoding="utf-8")

print(f"Script generation status: {script_generation_result['status']}")
print(f"Sections:                 {script_generation_result['summary']['section_count']}")
print(f"Words:                    {script_generation_result['summary']['total_word_count']}")
print(f"Estimated seconds:        {script_generation_result['summary']['estimated_duration_seconds']}")
print(f"Ready for verification:   {script_generation_result['handoff']['ready_for_verification']}")
print(f"Wrote:                    {SCRIPT_OUTPUT_DIR}")


RateLimitError: Error code: 429 - {'error': {'message': 'Request too large for gpt-4o in organization org-bH9828C8lFzDPOWWlIEO7iAG on tokens per min (TPM): Limit 30000, Requested 30972. The input or output tokens must be reduced in order to run successfully. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

In [ ]:
script_generation_result["summary"]


## App 6: Metadata Creator

In [ ]:
from src.pipeline_v2.apps.metadata_creator import run as run_metadata_creator

metadata_creator_input = build_metadata_creator_input(RUN_ID, script_generation_result, video_packaging_result)
metadata_creator_result = run_metadata_creator(metadata_creator_input)
require_app_ready(metadata_creator_result, "metadata_creator", handoff_key="ready_for_verification")
metadata_package = metadata_creator_result["data"]["metadata_package"]
youtube_metadata = metadata_package["youtube_metadata"]

write_json(METADATA_OUTPUT_DIR / "metadata_creator_result.json", metadata_creator_result)
write_json(METADATA_OUTPUT_DIR / "metadata_package.json", metadata_package)
write_json(METADATA_OUTPUT_DIR / "youtube_metadata.json", youtube_metadata)
write_json(METADATA_OUTPUT_DIR / "summary.json", metadata_creator_result["summary"])
write_jsonl(METADATA_OUTPUT_DIR / "llm_calls.jsonl", metadata_creator_result["llm_calls"])

print(f"Metadata status:          {metadata_creator_result['status']}")
print(f"Title options:            {metadata_creator_result['summary']['title_option_count']}")
print(f"Thumbnail options:        {metadata_creator_result['summary']['thumbnail_option_count']}")
print(f"Chapters:                 {metadata_creator_result['summary']['chapter_count']}")
print(f"Source URLs:              {metadata_creator_result['summary']['source_url_count']}")
print(f"Ready for verification:   {metadata_creator_result['handoff']['ready_for_verification']}")
print(f"Wrote:                    {METADATA_OUTPUT_DIR}")


In [ ]:
metadata_creator_result["summary"]